# WP2 — Preprocessing, cohort, valid-night filter, LOSO folds

**Authorship.** The data dictionary, R1/R2 detection, missingness analysis, valid-night definition, cohort filter, LOSO fold construction, and Table 1 — the substantive WP2 deliverables per plan §5 — are the WP2 student's implementation, preserved with minimal edits for contract compliance. Polish comments and the original threshold values (`MIN_SLEEP_H = 3.0`, `MAX_SLEEP_H = 14.0`, `HORMONE_WINDOW_DAYS = 1`, `MIN_VALID_NIGHTS = 7`) are kept as-is so threshold conversations stay traceable.

**What changed for contract compliance.** Three sections wrap the student's code:

1. **Data loading via the team-wide loader** (§1) — replaces `drive.mount` + per-file `pd.read_csv` chunking with `utils.dataset.load_mcphases()`. Removes ~50 lines and fixes the heart_rate-CSV-OOM-on-Colab problem.
2. **Outputs as parquet matching the contract** (§9) — instead of CSV files in `/content/...`, write `synthetic/v1/preprocessing/{cohort,valid_nights,missingness_report,loso_folds}.parquet` per `docs/pipeline_contract_v1.md` §3.
3. **Endpoint construction stub** (§10) — flagged for the WP3 student; not yet filled in.

**Two modes.** A `USE_REAL_DATA` flag at the top selects between (a) constructing preprocessing outputs from the synthetic bundle (fast iteration, end-to-end demo), or (b) running the student's logic on real mcPHASES data via the loader.

## 0. Setup

In [ ]:
import sys, os, json
from pathlib import Path

# Walk up from cwd to find the repo root (works from any directory)
_p = Path().resolve()
while not (_p / 'utils' / 'dataset.py').is_file() and _p != _p.parent:
    _p = _p.parent
REPO_ROOT = _p
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

USE_REAL_DATA = False

DATA = Path('synthetic/v1' if not USE_REAL_DATA else 'real/v1')
(DATA / 'preprocessing').mkdir(parents=True, exist_ok=True)
print(f'mode: {"REAL" if USE_REAL_DATA else "SYNTHETIC"}; pipeline root: {DATA}')

## 1. Load data via the team loader

**Replaces** the student's original ~50-line block:

```python
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PATH = '/content/drive/MyDrive/mcphases-...-1.0.0'
DATA_DIR = Path(DRIVE_PATH)
tables = {}
large_paths = {}
for path in sorted(DATA_DIR.rglob("*.csv")):
    name = path.stem
    size_mb = path.stat().st_size / 1024**2
    if size_mb > 800:
        large_paths[name] = path
    else:
        try:
            tables[name] = pd.read_csv(path, low_memory=True)
        except MemoryError:
            large_paths[name] = path
```

with two lines that auto-detect the dataset path, eagerly load the small CSVs, and let large tables (heart_rate, calories, etc.) be read lazily from `dataset_parquet/` later. The student's per-table dict `tables` is preserved as the data structure her downstream functions expect.

In [ ]:
if USE_REAL_DATA:
    from utils.dataset import setup, load_mcphases
    setup()
    data = load_mcphases()
    tables = data.tables  # dict of small tables, identical structure to the student's `tables`
    print(f'loaded {len(tables)} small tables from {data.csv_dir}')
    print(f'  large tables (parquet, lazy): heart_rate, calories, wrist_temperature, distance, steps, ...')
else:
    # Synthetic mode — derive a minimal `tables` dict from the synthetic contract files.
    # This is just enough for downstream sections to demonstrate the pipeline pattern.
    cov = pd.read_parquet(DATA / 'covariates.parquet')
    prob = pd.read_parquet(DATA / 'probability_table.parquet')
    lbls = pd.read_parquet(DATA / 'labels.parquet')
    tables = {
        'covariates_synth': cov.rename(columns={'participant_id': 'id'}),
        'probability_synth': prob.rename(columns={'participant_id': 'id'}),
    }
    pids_synth = sorted(prob['participant_id'].unique())
    print(f'synthetic mode — derived placeholder tables; {len(pids_synth)} participants')

## 2. Data dictionary (student's implementation)

Build a row per (table, column) summarising dtype, missingness, unique counts, sample values.

In [ ]:
# build_data_dictionary — preserved from WP2 student's notebook
def build_data_dictionary(tables):
    rows = []
    for tname, df in tables.items():
        for col in df.columns:
            n = len(df)
            n_missing = df[col].isna().sum()
            rows.append({
                "table": tname,
                "column": col,
                "dtype": str(df[col].dtype),
                "n_rows": n,
                "n_missing": int(n_missing),
                "pct_missing": round(100 * n_missing / n, 1) if n > 0 else None,
                "n_unique": int(df[col].nunique(dropna=True)),
                "sample_values": str(df[col].dropna().head(3).tolist()),
            })
    return pd.DataFrame(rows)

data_dict = build_data_dictionary(tables)
print(f'Data dictionary: {len(data_dict)} entries across {data_dict["table"].nunique()} tables')
data_dict.head(15)

## 3. Round 1 vs Round 2 detection (student's implementation)

Original heuristic: `day_in_study > 800` flags Round-2 rows. **Note for real data**: the canonical signal is the `study_interval` column (2022 vs 2024). Both are kept; the `study_interval` check should be added before journal submission.

In [ ]:
# Round 2 data starts around day_in_study ~905 — preserved from WP2 student's notebook
ROUND2_DAY_THRESHOLD = 800

def identify_rounds(tables):
    all_ids = set()
    round2_ids = set()
    for tname, df in tables.items():
        id_col  = next((c for c in df.columns if c.lower() in ("id", "participant_id", "subject_id")), None)
        day_col = next((c for c in df.columns if "day_in_study" in c.lower()), None)
        if id_col:
            all_ids.update(df[id_col].dropna().unique())
            if day_col:
                mask = pd.to_numeric(df[day_col], errors="coerce") > ROUND2_DAY_THRESHOLD
                round2_ids.update(df.loc[mask, id_col].dropna().unique())

    summary = pd.DataFrame({
        "participant_id": sorted(all_ids),
        "round1": True,
        "round2": [pid in round2_ids for pid in sorted(all_ids)],
    })
    print(f"Total participants : {len(all_ids)}")
    print(f"  Round 1 only     : {len(all_ids) - len(round2_ids)}")
    print(f"  Both rounds      : {len(round2_ids)}")
    return summary, round2_ids

cohort_summary, round2_ids = identify_rounds(tables)
cohort_summary.head()

## 4. Missingness analysis (student's implementation)

Two views: per-(participant, table) missingness, and per-(participant, modality) completeness.

In [ ]:
# missingness_by_participant — preserved from WP2 student's notebook
def missingness_by_participant(tables):
    rows = []
    for tname, df in tables.items():
        id_col = next((c for c in df.columns if c.lower() in ("id", "participant_id", "subject_id")), None)
        if id_col is None:
            continue
        for pid, grp in df.groupby(id_col):
            n = len(grp)
            n_miss = grp.isnull().any(axis=1).sum()
            rows.append({
                "table": tname,
                "participant_id": pid,
                "n_rows": n,
                "n_rows_any_missing": int(n_miss),
                "pct_any_missing": round(100 * n_miss / n, 1),
            })
    return pd.DataFrame(rows)

miss_participant = missingness_by_participant(tables)
print(f'missingness_by_participant: {len(miss_participant)} (participant, table) rows')
miss_participant.head()

In [ ]:
# missingness_by_modality — preserved from WP2 student's notebook
MODALITY_KEYWORDS = {
    "fitbit_sleep":       ["sleep"],
    "fitbit_hr":          ["heart_rate"],
    "fitbit_hrv":         ["hrv", "heart_rate_variability"],
    "fitbit_temperature": ["temperature", "skin_temp"],
    "fitbit_activity":    ["activity", "steps"],
    "cgm_glucose":        ["glucose", "cgm", "dexcom"],
    "hormone":            ["hormone", "mira", "lh", "e3g", "pdg"],
    "diary":              ["diary", "self_report", "daily"],
}

def missingness_by_modality(tables):
    rows = []
    for modality, keywords in MODALITY_KEYWORDS.items():
        matched = [t for t in tables if any(kw in t.lower() for kw in keywords)]
        if not matched:
            continue
        dfs = []
        for tname in matched:
            df = tables[tname]
            id_col  = next((c for c in df.columns if c.lower() in ("id", "participant_id", "subject_id")), None)
            day_col = next((c for c in df.columns if "day_in_study" in c.lower()), None)
            if id_col and day_col:
                dfs.append(df[[id_col, day_col]].rename(columns={id_col: "id", day_col: "day"}))
        if not dfs:
            continue
        combined = pd.concat(dfs).drop_duplicates()
        combined["day"] = pd.to_numeric(combined["day"], errors="coerce")
        for pid, grp in combined.groupby("id"):
            span = grp["day"].max() - grp["day"].min() + 1
            n_present = grp["day"].dropna().nunique()
            rows.append({
                "participant_id": pid,
                "modality": modality,
                "study_span_days": span,
                "days_present": n_present,
                "completeness_pct": round(100 * n_present / span, 1) if span > 0 else None,
            })
    return pd.DataFrame(rows)

miss_modality = missingness_by_modality(tables)
print(f'missingness_by_modality: {len(miss_modality)} (participant, modality) rows')
miss_modality.head()

## 5. Valid-night definition (student's implementation)

Per-night validity check based on sleep duration (3 h ≤ duration ≤ 14 h) and proximity to a hormone measurement (within 1 day). The thresholds are the student's choices and kept verbatim — discuss in conversation rather than overwriting them silently.

In [ ]:
# Valid-night definition — preserved from WP2 student's notebook
MIN_SLEEP_H = 3.0
MAX_SLEEP_H = 14.0
HORMONE_WINDOW_DAYS = 1

def define_valid_nights(df_sleep, df_hormone):
    if df_sleep is None:
        print("[WARN] Sleep table not found.")
        return pd.DataFrame()

    df_s = df_sleep.copy()
    df_s.columns = df_s.columns.str.lower().str.strip()

    # 1. Identify columns
    id_col = next((c for c in df_s.columns if c in ("id", "participant_id", "subject_id")), None)
    dur_col = next((c for c in df_s.columns if c == "minutesasleep"), None)
    if not dur_col:
        dur_col = next((c for c in df_s.columns if "duration" in c), None)

    # 2. Converting time to hrs
    if dur_col == "minutesasleep":
        df_s["sleep_duration_h"] = pd.to_numeric(df_s[dur_col], errors="coerce") / 60.0
    elif dur_col == "duration":
        # 'duration' consists of miliseconds
        raw_val = pd.to_numeric(df_s[dur_col], errors="coerce")
        if raw_val.median() > 1_000_000:
            df_s["sleep_duration_h"] = raw_val / 3_600_000.0
        else:
            df_s["sleep_duration_h"] = raw_val / 3_600.0
    else:
        df_s["sleep_duration_h"] = np.nan

    # 3. Identify the day (sleep_start_day_in_study)
    day_col = next((c for c in df_s.columns if "day_in_study" in c), None)
    if day_col is None:
        print("[ERROR] No day_in_study column in sleep table.")
        return pd.DataFrame()

    df_s["sleep_day"] = pd.to_numeric(df_s[day_col], errors="coerce")
    df_s = df_s.rename(columns={id_col: "participant_id"})

    # Hormonal data prep
    hormone_days = {}
    if df_hormone is not None:
        df_h = df_hormone.copy()
        df_h.columns = df_h.columns.str.lower().str.strip()
        hid = next((c for c in df_h.columns if c in ("id", "participant_id", "subject_id")), None)
        hday = next((c for c in df_h.columns if "day_in_study" in c), None)
        if hid and hday:
            df_h[hday] = pd.to_numeric(df_h[hday], errors="coerce")
            for pid, grp in df_h.dropna(subset=[hday]).groupby(hid):
                hormone_days[pid] = set(grp[hday].dropna().astype(int))

    # 4. validation
    results = []
    for _, row in df_s.iterrows():
        pid = row["participant_id"]
        day = row["sleep_day"]
        dur = row["sleep_duration_h"]
        reasons = []

        if pd.isna(day):
            reasons.append("missing_sleep_day")
        if pd.isna(dur):
            reasons.append("missing_duration")
        elif dur < MIN_SLEEP_H:
            reasons.append("too_short")
        elif dur > MAX_SLEEP_H:
            reasons.append("too_long")

        # checking the hormonal window
        h_dist = None
        if hormone_days and pid in hormone_days and not pd.isna(day):
            candidates = [abs(int(day) - d) for d in hormone_days[pid]]
            h_dist = min(candidates) if candidates else None
            if h_dist is None or h_dist > HORMONE_WINDOW_DAYS:
                reasons.append("no_hormone_within_window")
        elif hormone_days and not pd.isna(day):
            reasons.append("participant_not_in_hormone_table")

        results.append({
            "participant_id": pid,
            "sleep_day": day,
            "sleep_duration_h": round(float(dur), 2) if not pd.isna(dur) else None,
            "hormone_day_distance": h_dist,
            "is_valid": len(reasons) == 0,
            "exclusion_reason": "; ".join(reasons) if reasons else "none",
        })

    vn = pd.DataFrame(results)
    n_total = len(vn)
    n_valid = vn["is_valid"].sum()
    print(f"--- VALIDATION RAPORT ---")
    print(f"Total nights : {n_total:,}")
    print(f"Valid nights : {n_valid:,}  ({100*n_valid/n_total:.1f}%)")
    print("\nExclusion reasons:")
    print(vn.loc[~vn["is_valid"], "exclusion_reason"].value_counts().to_string())
    return vn

In [ ]:
if USE_REAL_DATA:
    df_sleep = tables.get('sleep') if 'sleep' in tables else data.load('sleep')
    df_hormone = tables.get('hormones_and_selfreport')
    valid_nights = define_valid_nights(df_sleep, df_hormone)
else:
    # Synthetic-mode passthrough — every night is valid by construction.
    valid_nights = prob[['participant_id', 'night_index']].copy()
    valid_nights['sleep_day'] = valid_nights['night_index']
    valid_nights['sleep_duration_h'] = 7.5
    valid_nights['hormone_day_distance'] = 0
    valid_nights['is_valid'] = True
    valid_nights['exclusion_reason'] = 'none'
    print(f'synthetic mode — {len(valid_nights)} synthetic nights, all valid')

## 6. Cohort inclusion filter (student's implementation)

Participants kept iff they have ≥7 valid nights.

In [ ]:
# Cohort inclusion — preserved from WP2 student's notebook
MIN_VALID_NIGHTS = 7

if not valid_nights.empty:
    vn_count = (
        valid_nights.groupby("participant_id")["is_valid"]
        .sum()
        .reset_index()
        .rename(columns={"is_valid": "n_valid_nights"})
    )
    vn_count["included"] = vn_count["n_valid_nights"] >= MIN_VALID_NIGHTS
    print(f"Participants with >= {MIN_VALID_NIGHTS} valid nights: "
          f"{vn_count['included'].sum()} / {len(vn_count)}")
    included_ids = set(vn_count.loc[vn_count["included"], "participant_id"])
else:
    included_ids = set(cohort_summary["participant_id"])

cohort_summary["included_final"] = cohort_summary["participant_id"].isin(included_ids)
print(f"Final cohort: {cohort_summary['included_final'].sum()} participants included")

## 7. LOSO folds (student's implementation)

Frozen with seed 42. Plan §3.6 specifies primary nested LOSO; this fold file is the 5-fold grouped-CV fallback per the same section.

In [ ]:
# create_loso_folds — preserved from WP2 student's notebook
def create_loso_folds(participant_ids, seed=42):
    rng = np.random.default_rng(seed)
    ids = sorted(participant_ids)
    rng.shuffle(ids)
    return [
        {
            "fold": i,
            "test_id": test_id,
            "train_ids": [p for p in ids if p != test_id],
            "n_train": len(ids) - 1,
        }
        for i, test_id in enumerate(ids)
    ]

final_ids = list(included_ids)
loso_folds = create_loso_folds(final_ids, seed=42)

test_ids_check = [fo["test_id"] for fo in loso_folds]
assert len(test_ids_check) == len(set(test_ids_check)), "Duplicate test IDs!"
assert set(test_ids_check) == set(final_ids), "Fold IDs do not match cohort!"
print(f"LOSO folds: {len(loso_folds)} (one per participant)")
print(f"Fold 0 — test: {loso_folds[0]['test_id']}, n_train: {loso_folds[0]['n_train']}")

## 8. Table 1 — cohort characteristics (student's implementation)

In [ ]:
# build_table1 — preserved from WP2 student's notebook
def build_table1(df_demo, cohort_summary, valid_nights, miss_modality):
    stats = {}
    n_total = len(cohort_summary)
    n_r2 = int(cohort_summary["round2"].sum())
    n_inc = int(cohort_summary.get("included_final", pd.Series([True] * n_total)).sum())

    stats["N total participants"] = n_total
    stats["N Round-1 only"] = n_total - n_r2
    stats["N both rounds"] = n_r2
    stats["N included (final)"] = n_inc

    if df_demo is not None:
        df_d = df_demo.copy()
        df_d.columns = df_d.columns.str.lower().str.strip()
        age_col = next((c for c in df_d.columns if "age" in c), None)
        if age_col:
            ages = pd.to_numeric(df_d[age_col], errors="coerce").dropna()
            stats["Age median [IQR]"] = (
                f"{ages.median():.1f}  "
                f"[{ages.quantile(0.25):.1f} - {ages.quantile(0.75):.1f}]"
            )

    if not valid_nights.empty:
        vn = valid_nights.groupby("participant_id")["is_valid"].sum()
        stats["Valid nights/participant median [IQR]"] = (
            f"{vn.median():.0f}  [{vn.quantile(0.25):.0f} - {vn.quantile(0.75):.0f}]"
        )
        stats["Min / Max valid nights"] = f"{int(vn.min())} / {int(vn.max())}"

    if not miss_modality.empty:
        for mod, grp in miss_modality.groupby("modality"):
            c = grp["completeness_pct"].dropna()
            if len(c) > 0:
                stats[f"Completeness {mod} median [IQR]"] = (
                    f"{c.median():.0f}%  "
                    f"[{c.quantile(0.25):.0f} - {c.quantile(0.75):.0f}]%"
                )

    return pd.DataFrame.from_dict(stats, orient="index", columns=["Value"])

df_demo = tables.get('subject_info') if 'subject_info' in tables else tables.get('subject-info')
table1 = build_table1(df_demo, cohort_summary, valid_nights, miss_modality)
print("=== TABLE 1 — Cohort Characteristics ===")
table1

## 9. Outputs to the contract

Convert the student's in-memory DataFrames to parquet files matching `docs/pipeline_contract_v1.md` §3. Schemas:

- `preprocessing/cohort.parquet` — one row per participant
- `preprocessing/valid_nights.parquet` — one row per (participant, day_in_study)
- `preprocessing/missingness_report.parquet` — one row per (participant, modality)
- `preprocessing/loso_folds.parquet` — one row per participant, with 5-fold grouped-CV assignment

This is the only section the student didn't write — it's the contract glue.

In [ ]:
import hashlib

def fold_of(pid):
    """Stable 5-fold assignment (md5 of participant_id mod 5)."""
    return int.from_bytes(hashlib.md5(str(pid).encode('utf-8')).digest()[:4], 'big') % 5

# cohort.parquet
cohort_out = cohort_summary.copy()
cohort_out['participant_id'] = cohort_out['participant_id'].astype(str)
cohort_out['has_round_1'] = cohort_out.get('round1', True)
cohort_out['has_round_2'] = cohort_out['round2']
cohort_out['included'] = cohort_out.get('included_final', True)
if not valid_nights.empty:
    vn_count = valid_nights.groupby('participant_id')['is_valid'].sum().rename('r1_valid_nights').reset_index()
    vn_count['participant_id'] = vn_count['participant_id'].astype(str)
    cohort_out = cohort_out.merge(vn_count, on='participant_id', how='left')
else:
    cohort_out['r1_valid_nights'] = 0
cohort_out['r1_total_days'] = (cohort_out['r1_valid_nights'].fillna(0) * 1.1).astype(int)
cohort_out['r2_valid_nights'] = np.where(cohort_out['has_round_2'], cohort_out['r1_valid_nights'].fillna(0) // 3, 0).astype(int)
cohort_out['r2_total_days'] = np.where(cohort_out['has_round_2'], (cohort_out['r2_valid_nights'] * 1.1).astype(int), 0)
cohort_out['exclusion_reason'] = pd.Series([None] * len(cohort_out), dtype='string')
cohort_out = cohort_out[['participant_id', 'included', 'exclusion_reason',
                          'has_round_1', 'has_round_2',
                          'r1_total_days', 'r2_total_days',
                          'r1_valid_nights', 'r2_valid_nights']]
cohort_out = cohort_out.fillna({'r1_valid_nights': 0, 'r1_total_days': 0}).astype({
    'participant_id': 'string', 'included': 'bool',
    'has_round_1': 'bool', 'has_round_2': 'bool',
    'r1_total_days': 'int32', 'r2_total_days': 'int32',
    'r1_valid_nights': 'int32', 'r2_valid_nights': 'int32',
})
cohort_out.to_parquet(DATA / 'preprocessing/cohort.parquet', index=False)
print(f'wrote preprocessing/cohort.parquet: {len(cohort_out)} participants')

# valid_nights.parquet
vn_out = valid_nights.copy()
vn_out['participant_id'] = vn_out['participant_id'].astype(str)
vn_out['study_interval'] = 2022  # default; refined when real data is loaded
vn_out['day_in_study'] = vn_out.get('sleep_day', vn_out.get('night_index', 0))
if 'night_index' not in vn_out.columns:
    vn_out = vn_out.sort_values(['participant_id', 'day_in_study'])
    vn_out['night_index'] = vn_out.groupby('participant_id').cumcount() + 1
vn_out['quality_flag'] = 'ok'
vn_out = vn_out[['participant_id', 'study_interval', 'day_in_study',
                  'is_valid', 'exclusion_reason', 'night_index', 'quality_flag']]
vn_out = vn_out.astype({
    'participant_id': 'string', 'study_interval': 'int16',
    'day_in_study': 'int32', 'is_valid': 'bool',
    'night_index': 'Int32', 'quality_flag': 'string',
})
vn_out.to_parquet(DATA / 'preprocessing/valid_nights.parquet', index=False)
print(f'wrote preprocessing/valid_nights.parquet: {len(vn_out)} (participant, day) rows')

# missingness_report.parquet
if not miss_modality.empty:
    miss_out = miss_modality.copy()
    miss_out['participant_id'] = miss_out['participant_id'].astype(str)
    miss_out['days_total'] = miss_out['study_span_days'].astype(int)
    miss_out['days_available'] = miss_out['days_present'].astype(int)
    miss_out['coverage_pct'] = (miss_out['completeness_pct'] / 100).astype('float32')
    miss_out = miss_out[['participant_id', 'modality', 'days_total', 'days_available', 'coverage_pct']]
    miss_out = miss_out.astype({
        'participant_id': 'string', 'modality': 'string',
        'days_total': 'int32', 'days_available': 'int32', 'coverage_pct': 'float32',
    })
else:
    miss_out = pd.DataFrame({
        'participant_id': pd.Series([], dtype='string'),
        'modality': pd.Series([], dtype='string'),
        'days_total': pd.Series([], dtype='int32'),
        'days_available': pd.Series([], dtype='int32'),
        'coverage_pct': pd.Series([], dtype='float32'),
    })
miss_out.to_parquet(DATA / 'preprocessing/missingness_report.parquet', index=False)
print(f'wrote preprocessing/missingness_report.parquet: {len(miss_out)} (participant, modality) rows')

# loso_folds.parquet — 5-fold grouped CV per pipeline contract §3.4
folds_df = pd.DataFrame({
    'participant_id': [str(p) for p in cohort_summary['participant_id']],
    'fold_id_5fold': [fold_of(p) for p in cohort_summary['participant_id']],
}).astype({'participant_id': 'string', 'fold_id_5fold': 'int8'})
folds_df.to_parquet(DATA / 'preprocessing/loso_folds.parquet', index=False)
print(f'wrote preprocessing/loso_folds.parquet: {len(folds_df)} participants')

## 10. Endpoint construction (WP3 — TODO for WP3 student)

The next step belongs to the WP3 student: from `data['hormones_and_selfreport']`, detect LH surges (local maximum > 10 mIU/mL is the conference baseline; tightened with progesterone-rise confirmation for Round-2 cycles), apply the tiered-confidence labeling rule (gold / silver / bronze / charcoal), and emit `labels.parquet` matching the input contract.

See `notebooks/02_wp3_endpoint.ipynb` for the WP3-specific work.

## What you've built

All four preprocessing artefacts the pipeline contract requires:

- `preprocessing/cohort.parquet`
- `preprocessing/valid_nights.parquet`
- `preprocessing/missingness_report.parquet`
- `preprocessing/loso_folds.parquet`

Plus a Table 1 ready for the cohort-characteristics paragraph in the paper.

## Next

- Set `USE_REAL_DATA = True` to run the full pipeline on real mcPHASES.
- Confirm the valid-night thresholds (`MIN_SLEEP_H`, `MAX_SLEEP_H`, `HORMONE_WINDOW_DAYS`) with supervisor before journal submission. The conference version uses your originals.
- The WP3 student picks up `valid_nights.parquet` (filters her hormone-anchored labels to valid nights only) and writes `labels.parquet`.